# TweetNLP tutorial (cardiffnlp) adapted to embed our tweets with `cardiffnlp/tweet-topic-latest-multi` and train CatBoost

Based on the official TweetNLP introduction notebook: https://github.com/cardiffnlp/tweetnlp

# TweetNLP Introduction
This colab notebook brings a short introduction of [`tweetnlp`](https://github.com/cardiffnlp/tweetnlp), a python library of NLP models for tweets. In this tutorial, we explain following applications on tweets:
- [Text Classification](#scrollTo=KAZYjeskBqL4): Sentiment/Hate/Irony/Emoji/Emotion, etc
- [NER](#scrollTo=WeREiLEjBlrj): Named Entity Recognition (NER)
- [Question Answering](#scrollTo=reZDePaBmYhA&line=4&uniqifier=1): Answer prediction given a question with a context (SQuAD style)
- [Question Answer Generation](#scrollTo=uqd7sBHhnwym&line=6&uniqifier=1): Question and answer pairs generation on a context
- [Language Modeling](#scrollTo=COOoZHVAFCIG): Masked token prediction
- [Fine-tuning](#scrollTo=2plrPTqk7OHp): Model fine-tuning.


## Installation
TweetNLP is available on pip or can be installed from source.


In [ ]:
DATA_DIR = "../data"

In [1]:
# Fix Colab Error
!pip install --upgrade google-cloud-storage

In [2]:
# via pip
!pip install tweetnlp

In [3]:
# # via source
# !git clone https://github.com/cardiffnlp/tweetnlp
# %cd tweetnlp
# !pip install . -U

In [4]:
! pip list | grep tweetnlp

All you need is to import `tweetnlp` !

In [5]:
import tweetnlp

## Tweet Classification
The classification module consists of six different tasks (Topic Classification, Sentiment Analysis, Irony Detection, Hate Speech Detection, Offensive Language Detection, Emoji Prediction, and Emotion Analysis).
In each example, the model is instantiated by `tweetnlp.load("task-name")`, and run the prediction by passing a text or a list of texts as argument to the corresponding function.

### Topic Classification
The aim of this task is, given a tweet to assign topics related to its content. The task is formed as a supervised multi-label classification problem where each tweet is assigned one or more topics from a total of 19 available topics. The topics were carefully curated based on Twitter trends with the aim to be broad and general and consist of classes such as: arts and culture, music, or sports. Our internally-annotated dataset contains over 10K manually-labeled tweets (check the paper [here](https://arxiv.org/abs/2209.09824), or the [huggingface dataset page](https://huggingface.co/datasets/cardiffnlp/tweet_topic_single)).

***Multi-label Model***

In [ ]:
model = tweetnlp.load_model('topic_classification')  # Or `model = tweetnlp.TopicClassification()`
model.topic("Jacob Collier is a Grammy-awarded English artist from London.")  # Or `model.predict`

{'label': ['celebrity_&_pop_culture', 'music']}

In [ ]:
# Note: the probability of the multi-label model is the output of sigmoid function on binary prediction whether each topic is positive or negative.
model.topic("Jacob Collier is a Grammy-awarded English artist from London.", return_probability=True)

{'label': ['celebrity_&_pop_culture', 'music'],
 'probability': {'arts_&_culture': 0.03737165033817291,
  'business_&_entrepreneurs': 0.010188567452132702,
  'celebrity_&_pop_culture': 0.9244890213012695,
  'diaries_&_daily_life': 0.03425709903240204,
  'family': 0.007961377501487732,
  'fashion_&_style': 0.020642098039388657,
  'film_tv_&_video': 0.08062593638896942,
  'fitness_&_health': 0.0063430978916585445,
  'food_&_dining': 0.004288368858397007,
  'gaming': 0.004327302798628807,
  'learning_&_educational': 0.010652048513293266,
  'music': 0.8291940689086914,
  'news_&_social_concern': 0.24688191711902618,
  'other_hobbies': 0.02067117765545845,
  'relationships': 0.020371070131659508,
  'science_&_technology': 0.017007505521178246,
  'sports': 0.01429108064621687,
  'travel_&_adventure': 0.010423889383673668,
  'youth_&_student_life': 0.008605164475739002}}

***Singlelabel Model***

In [ ]:
model = tweetnlp.load_model('topic_classification', multi_label=False)  # Or `model = tweetnlp.TopicClassification(multi_label=False)`
model.topic("Jacob Collier is a Grammy-awarded English artist from London.")

{'label': 'pop_culture'}

In [ ]:
# NOTE: the probability of the sinlge-label model the softmax over the label.
model.topic("Jacob Collier is a Grammy-awarded English artist from London.", return_probability=True)

{'label': 'pop_culture',
 'probability': {'arts_&_culture': 9.20625461731106e-05,
  'business_&_entrepreneurs': 6.916998972883448e-05,
  'pop_culture': 0.9995898604393005,
  'daily_life': 0.00011083026038249955,
  'sports_&_gaming': 8.668467489769682e-05,
  'science_&_technology': 5.152115045348182e-05}}

***Dataset***

In [ ]:
dataset_multi_label, label2id_multi_label = tweetnlp.load_dataset('topic_classification')
dataset_single_label, label2id_single_label = tweetnlp.load_dataset('topic_classification', multi_label=False)

In [ ]:
dataset_multi_label

DatasetDict({
    test_2020: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 376
    })
    test_2021: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 1693
    })
    train_2020: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 2858
    })
    train_2021: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 1516
    })
    train_all: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 4374
    })
    validation_2020: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 352
    })
    validation_2021: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 189
    })
    train_random: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 2830
    })
    validati

In [ ]:
label2id_multi_label

{'arts_&_culture': 0,
 'business_&_entrepreneurs': 1,
 'pop_culture': 2,
 'daily_life': 3,
 'sports_&_gaming': 4,
 'science_&_technology': 5}

In [ ]:
dataset_single_label

DatasetDict({
    test_2020: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 376
    })
    test_2021: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 1693
    })
    train_2020: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 2858
    })
    train_2021: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 1516
    })
    train_all: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 4374
    })
    validation_2020: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 352
    })
    validation_2021: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 189
    })
    train_random: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 2830
    })
    validati

In [ ]:
label2id_single_label

{'arts_&_culture': 0,
 'business_&_entrepreneurs': 1,
 'pop_culture': 2,
 'daily_life': 3,
 'sports_&_gaming': 4,
 'science_&_technology': 5}

### Sentiment Analysis
The sentiment analysis task integrated in TweetNLP is a simplified version where the goal is to predict the sentiment of a tweet with one of the three following labels: positive, neutral or negative. The base dataset for English is the unified TweetEval version of the Semeval-2017 dataset from the task on Sentiment Analysis in Twitter (check the paper [here](https://arxiv.org/pdf/2010.12421.pdf)).

***English Model***

In [ ]:
model = tweetnlp.load_model('sentiment')  # Or `model = tweetnlp.Sentiment()`
model.sentiment("Yes, including Medicare and social security saving👍")  # Or `model.predict`

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


{'label': 'positive'}

In [ ]:
model.sentiment("Yes, including Medicare and social security saving👍", return_probability=True)

{'label': 'positive',
 'probability': {'negative': 0.004584962967783213,
  'neutral': 0.19360849261283875,
  'positive': 0.8018065094947815}}

***Multilingual Model***

In [ ]:
model = tweetnlp.load_model('sentiment', multilingual=True)  # Or `model = tweetnlp.Sentiment(multilingual=True)`
model.sentiment("天気が良いとやっぱり気持ち良いなあ✨")

{'label': 'positive'}

In [ ]:
model.sentiment("天気が良いとやっぱり気持ち良いなあ✨", return_probability=True)

{'label': 'positive',
 'probability': {'negative': 0.028369618579745293,
  'neutral': 0.08128826320171356,
  'positive': 0.8903420567512512}}

***Dataset***

In [ ]:
dataset, label2id = tweetnlp.load_dataset('sentiment')

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [ ]:
label2id

{'negative': 0, 'neutral': 1, 'positive': 2}

In [ ]:
for l in ['arabic', 'english', 'french', 'german', 'hindi', 'italian', 'portuguese', 'spanish']:
    dataset_multilingual, label2id_multilingual = tweetnlp.load_dataset('sentiment', multilingual=True, task_language=l)
    print(dataset_multilingual)
    print(label2id_multilingual)
    print()

  0%|          | 0/3 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1839
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 324
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 870
    })
})
{'negative': 0, 'neutral': 1, 'positive': 2}



  0%|          | 0/3 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1839
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 324
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 870
    })
})
{'negative': 0, 'neutral': 1, 'positive': 2}



  0%|          | 0/3 [00:00<?, ?it/s]

...

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1839
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 324
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 870
    })
})
{'negative': 0, 'neutral': 1, 'positive': 2}



  0%|          | 0/3 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1839
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 324
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 870
    })
})
{'negative': 0, 'neutral': 1, 'positive': 2}



  0%|          | 0/3 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1839
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 324
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 870
    })
})
{'negative': 0, 'neutral': 1, 'positive': 2}



### Irony Detection
This is a binary classification task where given a tweet, the goal is to detect whether it is ironic or not. It is based on the Irony Detection dataset from the SemEval 2018 task (check the paper [here](https://arxiv.org/pdf/2010.12421.pdf)).

***Model***

In [ ]:
model = tweetnlp.load_model('irony')  # Or `model = tweetnlp.Irony()`
model.irony('If you wanna look like a badass, have drama on social media')  # Or `model.predict`

{'label': 'irony'}

In [ ]:
model.irony('If you wanna look like a badass, have drama on social media', return_probability=True)

{'label': 'irony',
 'probability': {'non_irony': 0.08390878140926361,
  'irony': 0.9160912036895752}}

***Dataset***

In [ ]:
dataset, label2id = tweetnlp.load_dataset('irony')

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 2862
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 784
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 955
    })
})

In [ ]:
label2id

{'non_irony': 0, 'irony': 1}

### Hate Speech Detection
The hate speech dataset consists of detecting whether a tweet is hateful towards women or immigrants. It is based on the Detection of Hate Speech task at SemEval 2019 (check the paper [here](https://arxiv.org/pdf/2010.12421.pdf)).

***Model***

In [ ]:
model = tweetnlp.load_model('hate')  # Or `model = tweetnlp.Hate()`
model.hate('Whoever just unfollowed me you a bitch')  # Or `model.predict`

{'label': 'non-hate'}

In [ ]:
model.hate('Whoever just unfollowed me you a bitch', return_probability=True)

{'label': 'non-hate',
 'probability': {'non-hate': 0.726382851600647, 'hate': 0.27361705899238586}}

***Dataset***

In [ ]:
dataset, label2id = tweetnlp.load_dataset('hate')

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2970
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1000
    })
})

In [ ]:
label2id

{'non-hate': 0, 'hate': 1}

### Offensive Language Identification
This task consists in identifying whether some form of offensive language is present in a tweet. For our benchmark we rely on the SemEval2019 OffensEval dataset (check the paper [here](https://arxiv.org/pdf/2010.12421.pdf)).

***Model***

In [ ]:
model = tweetnlp.load_model('offensive')  # Or `model = tweetnlp.Offensive()`
model.offensive("All two of them taste like ass.")  # Or `model.predict`

{'label': 'offensive'}

In [ ]:
model.offensive("All two of them taste like ass.", return_probability=True)

{'label': 'offensive',
 'probability': {'non-offensive': 0.16420336067676544,
  'offensive': 0.8357967138290405}}

***Dataset***

In [ ]:
dataset, label2id = tweetnlp.load_dataset('offensive')

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 11916
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 860
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1324
    })
})

In [ ]:
label2id

{'non-offensive': 0, 'offensive': 1}

### Emoji Prediction
The goal of emoji prediction is to predict the final emoji on a given tweet. The dataset used to fine-tune our models is the TweetEval adaptation from the SemEval 2018 task on Emoji Prediction (check the paper [here](https://arxiv.org/pdf/2010.12421.pdf)), including 20 emoji as labels (❤, 😍, 😂, 💕, 🔥, 😊, 😎, ✨, 💙, 😘, 📷, 🇺🇸, ☀, 💜, 😉, 💯, 😁, 🎄, 📸, 😜).

***Model***

In [ ]:
model = tweetnlp.load_model('emoji')  # Or `model = tweetnlp.Emoji()`
model.emoji('Beautiful sunset last night from the pontoon @TupperLakeNY')  # Or `model.predict`

{'label': '📷'}

In [ ]:
model.emoji('Beautiful sunset last night from the pontoon @TupperLakeNY', return_probability=True)

{'label': '📷',
 'probability': {'❤': 0.13197320699691772,
  '😍': 0.11246418952941895,
  '😂': 0.008415068499743938,
  '💕': 0.04842923954129219,
  '🔥': 0.014528144150972366,
  '😊': 0.15096743404865265,
  '😎': 0.08625395596027374,
  '✨': 0.01616634987294674,
  '💙': 0.07396604120731354,
  '😘': 0.03033280000090599,
  '📷': 0.1652531772851944,
  '🇺🇸': 0.020336609333753586,
  '☀': 0.007999817840754986,
  '💜': 0.01611141860485077,
  '😉': 0.012984543107450008,
  '💯': 0.012557175941765308,
  '😁': 0.0313868410885334,
  '🎄': 0.006829544901847839,
  '📸': 0.04188750311732292,
  '😜': 0.01115693524479866}}

***Dataset***

In [ ]:
dataset, label2id = tweetnlp.load_dataset('emoji')

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
})

In [ ]:
label2id

{'❤': 0,
 '😍': 1,
 '😂': 2,
 '💕': 3,
 '🔥': 4,
 '😊': 5,
 '😎': 6,
 '✨': 7,
 '💙': 8,
 '😘': 9,
 '📷': 10,
 '🇺🇸': 11,
 '☀': 12,
 '💜': 13,
 '😉': 14,
 '💯': 15,
 '😁': 16,
 '🎄': 17,
 '📸': 18,
 '😜': 19}

### Emotion Recognition
Given a tweet, this task consists of associating it with its most appropriate emotion. As a reference dataset we use the SemEval 2018 task on Affect in Tweets, simplified to only four emotions used in TweetEval: anger, joy, sadness and optimism (check the paper [here](https://arxiv.org/pdf/2010.12421.pdf)).

***Model***

In [ ]:
model = tweetnlp.load_model('emotion')  # Or `model = tweetnlp.Emotion()`
model.emotion('I love swimming for the same reason I love meditating...the feeling of weightlessness.')  # Or `model.predict`

{'label': 'optimism'}

In [ ]:
model.emotion('I love swimming for the same reason I love meditating...the feeling of weightlessness.', return_probability=True)

{'label': 'optimism',
 'probability': {'joy': 0.013675869442522526,
  'optimism': 0.7345258593559265,
  'anger': 0.17707151174545288,
  'sadness': 0.07472679018974304}}

***Dataset***

In [ ]:
dataset, label2id = tweetnlp.load_dataset('emotion')

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 3257
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1421
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 374
    })
})

In [ ]:
label2id

{'anger': 0, 'joy': 1, 'optimism': 2, 'sadness': 3}

## Named Entity Recognition
This module consists of a named-entity recognition (NER) model specifically trained for tweets. The model is instantiated by `tweetnlp.load("ner")`, and runs the prediction by giving a text or a list of texts as argument to the `ner` function (check the paper [here](https://arxiv.org/abs/2210.03797), or the [huggingface dataset page](https://huggingface.co/datasets/tner/tweetner7)).

***Model***

In [ ]:
model = tweetnlp.load_model('ner')  # Or `model = tweetnlp.NER()`
model.ner('Jacob Collier is a Grammy-awarded English artist from London.')  # Or `model.predict`

[{'type': 'person', 'entity': 'Jacob Collier'},
 {'type': 'event', 'entity': ' Grammy'},
 {'type': 'location', 'entity': ' London'}]

In [ ]:
# Note: the probability for the predicted entity is the mean of the probabilities over the sub-tokens representing the entity.
model.ner('Jacob Collier is a Grammy-awarded English artist from London.', return_probability=True)  # Or `model.predict`

[{'type': 'person',
  'entity': 'Jacob Collier',
  'probability': 0.9905317823092142},
 {'type': 'event', 'entity': ' Grammy', 'probability': 0.1916438639163971},
 {'type': 'location', 'entity': ' London', 'probability': 0.9606999158859253}]

***Dataset***

In [ ]:
dataset, label2id = tweetnlp.load_dataset('ner')

In [ ]:
dataset

DatasetDict({
    test_2020: Dataset({
        features: ['tokens', 'tags', 'id', 'date'],
        num_rows: 576
    })
    test_2021: Dataset({
        features: ['tokens', 'tags', 'id', 'date'],
        num_rows: 2807
    })
    validation_2020: Dataset({
        features: ['tokens', 'tags', 'id', 'date'],
        num_rows: 576
    })
    validation_2021: Dataset({
        features: ['tokens', 'tags', 'id', 'date'],
        num_rows: 310
    })
    train_2020: Dataset({
        features: ['tokens', 'tags', 'id', 'date'],
        num_rows: 4616
    })
    train_2021: Dataset({
        features: ['tokens', 'tags', 'id', 'date'],
        num_rows: 2495
    })
    train_all: Dataset({
        features: ['tokens', 'tags', 'id', 'date'],
        num_rows: 7111
    })
    validation_random: Dataset({
        features: ['tokens', 'tags', 'id', 'date'],
        num_rows: 576
    })
    train_random: Dataset({
        features: ['tokens', 'tags', 'id', 'date'],
        num_rows: 4616
    })
  

In [ ]:
label2id

{'B-corporation': 0,
 'B-creative_work': 1,
 'B-event': 2,
 'B-group': 3,
 'B-location': 4,
 'B-person': 5,
 'B-product': 6,
 'I-corporation': 7,
 'I-creative_work': 8,
 'I-event': 9,
 'I-group': 10,
 'I-location': 11,
 'I-person': 12,
 'I-product': 13,
 'O': 14}

## Question Answering
This module consists of a question answering model specifically trained for tweets.
The model is instantiated by `tweetnlp.load("question_answering")`,
and runs the prediction by giving a question or a list of questions along with a context or a list of contexts
as argument to the `question_answering` function (check the paper [here](https://arxiv.org/abs/2210.03992), or the [huggingface dataset page](https://huggingface.co/datasets/lmqg/qg_tweetqa)).

***Model***

In [ ]:
model = tweetnlp.load_model('question_answering')  # Or `model = tweetnlp.QuestionAnswering()`
model.question_answering(
  question='who created the post as we know it today?',
  context="'So much of The Post is Ben,' Mrs. Graham said in 1994, three years after Bradlee retired as editor. 'He created it as we know it today.'— Ed O'Keefe (@edatpost) October 21, 2014"
)  # Or `model.predict`

{'generated_text': 'nebraska'}

***Dataset***

In [ ]:
dataset = tweetnlp.load_dataset('question_answering')

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['answer', 'paragraph_question', 'question', 'paragraph'],
        num_rows: 9489
    })
    validation: Dataset({
        features: ['answer', 'paragraph_question', 'question', 'paragraph'],
        num_rows: 1086
    })
    test: Dataset({
        features: ['answer', 'paragraph_question', 'question', 'paragraph'],
        num_rows: 1203
    })
})

## Question Answer Generation
This module consists of a question & answer pair generation specifically trained for tweets.
The model is instantiated by `tweetnlp.load("question_answer_generation")`,
and runs the prediction by giving a context or a list of contexts
as argument to the `question_answer_generation` function (check the paper [here](https://arxiv.org/abs/2210.03992), or the [huggingface dataset page](https://huggingface.co/datasets/lmqg/qag_tweetqa)).


***Model***

In [ ]:
model = tweetnlp.load_model('question_answer_generation')  # Or `model = tweetnlp.QuestionAnswerGeneration()`
model.question_answer_generation(
  text="'So much of The Post is Ben,' Mrs. Graham said in 1994, three years after Bradlee retired as editor. 'He created it as we know it today.'— Ed O'Keefe (@edatpost) October 21, 2014"
)  # Or `model.predict`

[{'question': 'who created the post?', 'answer': 'ben'},
 {'question': 'what did ben do in 1994?', 'answer': 'he retired as editor'}]

***Dataset***

In [ ]:
dataset = tweetnlp.load_dataset("question_answer_generation")

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['answers', 'questions', 'paragraph', 'paragraph_id', 'questions_answers'],
        num_rows: 4536
    })
    validation: Dataset({
        features: ['answers', 'questions', 'paragraph', 'paragraph_id', 'questions_answers'],
        num_rows: 583
    })
    test: Dataset({
        features: ['answers', 'questions', 'paragraph', 'paragraph_id', 'questions_answers'],
        num_rows: 583
    })
})

## Language Modeling
The masked language model predicts the masked token in the given sentence. This is instantiated by `tweetnlp.load('language_model')`, and runs the prediction by giving a text or a list of texts as argument to the `mask_prediction` function. Please make sure that each text has a `<mask>` token, since that is eventually the following by the objective of the model to predict.

In [ ]:
model = tweetnlp.load_model('language_model')  # Or `model = tweetnlp.LanguageModel()`
model.mask_prediction("So glad I'm <mask> vaccinated.")  # Or `model.predict`

Downloading:   0%|          | 0.00/798k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/456k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/239 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/499M [00:00<?, ?B/s]

{'best_tokens': ['fully',
  'getting',
  'not',
  'still',
  'already',
  'all',
  'being',
  'completely',
  'now',
  'finally'],
 'best_scores': [1.0366170595521584e-11,
  1.6352504420003022e-11,
  1.9819770824547334e-11,
  4.520027963028639e-10,
  1.0578210094536189e-05,
  0.0002495304506737739,
  2.3952467017807066e-05,
  1.8536340576247312e-05,
  2.8879132514703088e-05,
  5.781193976872601e-06],
 'best_sentences': ["So glad I'm fully vaccinated.",
  "So glad I'm getting vaccinated.",
  "So glad I'm not vaccinated.",
  "So glad I'm still vaccinated.",
  "So glad I'm already vaccinated.",
  "So glad I'm all vaccinated.",
  "So glad I'm being vaccinated.",
  "So glad I'm completely vaccinated.",
  "So glad I'm now vaccinated.",
  "So glad I'm finally vaccinated."]}

## Tweet Embedding
The tweet embedding model produces a fixed length embedding for a tweet. The embedding represents the semantics by meaning of the tweet, and this can be used for semantic search of tweets by using the similarity between the embeddings. Model is instantiated by `tweet_nlp.load('sentence_embedding')`, and run the prediction by passing a text or a list of texts as argument to the `embedding` function.

In [16]:
import pandas as pd
import numpy as np

In [9]:
def remove_noise_eng(text):
    text = str(text)
    text = text.replace("&amp;", "and")
    text = text.replace("​", "")
    text = text.strip()
    return text

In [10]:
train_df = pd.read_csv(f"{DATA_DIR}/train_eng_v2.csv")
train_df.head()

                                                text             label
0  Prabowo's visit was to inaugurate and hand ove...  Sumber Daya Alam
1  Anies received a standing ovation when he beca...           Politik
2  It's true, there are supporters of 01 who are ...         Demografi
3  When Anies was critical of Pak Prabowo's perfo...           Politik
4  Anies Baswedan hopes that ASN, including the T...           Politik

In [11]:
train_df["text"] = train_df["text"].apply(remove_noise_eng)
train_df.head()

                                                text             label
0  Prabowo's visit was to inaugurate and hand ove...  Sumber Daya Alam
1  Anies received a standing ovation when he beca...           Politik
2  It's true, there are supporters of 01 who are ...         Demografi
3  When Anies was critical of Pak Prabowo's perfo...           Politik
4  Anies Baswedan hopes that ASN, including the T...           Politik

In [36]:
# model = tweetnlp.load_model('sentence_embedding')
model = tweetnlp.load_model('sentence_embedding', model_name='cardiffnlp/tweet-topic-latest-multi')
# model = tweetnlp.load_model('sentiment', model_name='cardiffnlp/twitter-roberta-base-sentiment-latest')

Some weights of the model checkpoint at /root/.cache/torch/sentence_transformers/cardiffnlp_tweet-topic-latest-multi were not used when initializing RobertaModel: ['classifier.dense.bias', 'classifier.out_proj.weight', 'classifier.dense.weight', 'classifier.out_proj.bias']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaModel were not initialized from the model checkpoint at /root/.cache/torch/sentence_transformers/cardiffnlp_tweet-topic-latest-multi and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRA

In [37]:
vectors = []
for i in range(len(train_df)):
  vectors.append(model.embedding(train_df["text"].iloc[i]))
vectors = np.array(vectors)
vectors.shape

(4218, 768)

In [38]:
train_df_vectors = pd.DataFrame(vectors)
train_df_vectors.head()

        0         1         2         3         4         5         6    \
0  0.141808 -0.701733 -0.700338  0.750851  2.065928  0.177354 -0.231075   
1 -0.299084 -0.814394 -0.715309  0.832515  1.533038 -0.201577  0.063790   
2 -0.098241 -0.223466 -0.615167  0.891725  0.835152  0.391385 -0.796614   
3 -0.191056  0.133321 -0.408255 -0.260493  0.399769 -0.152812 -0.034994   
4 -0.121173 -1.052810 -0.198817  1.259770  0.315285 -0.226046  0.350928   

        7         8         9    ...       758       759       760       761  \
0  0.591776 -0.921538 -1.029029  ...  0.795434 -0.164769 -0.951676  0.129432   
1  1.378057 -0.266698 -0.636646  ...  0.209888 -0.336658 -0.880570  0.352392   
2  1.191605 -0.253336 -0.765960  ... -0.062412 -0.000958  0.373149  0.116801   
3  0.957224 -0.454925 -0.710336  ... -0.218519 -0.011416 -0.307208 -0.150195   
4  1.231895 -0.214026 -0.484378  ...  0.498048 -0.067719 -1.192021  0.355567   

        762       763       764       765       766       767  
0  1

In [44]:
train_df_vectors.to_csv(f"{DATA_DIR}/train_tweet_nlp_embedding_v2.csv")

In [39]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(train_df_vectors, train_df['label'], test_size=0.2, random_state=42)

X_train.shape, X_val.shape, y_train.shape, y_val.shape

((3374, 768), (844, 768), (3374,), (844,))

In [ ]:
!pip install catboost

In [40]:
from catboost import CatBoostClassifier

clf = CatBoostClassifier(
    task_type="GPU",
    verbose=100,
    loss_function='MultiClass',
    eval_metric='TotalF1',
    random_state=42
)
clf.fit(
    X_train, y_train,
    eval_set=(X_val, y_val)
)

Learning rate set to 0.118898
0:	learn: 0.6229904	test: 0.6088749	best: 0.6088749 (0)	total: 76.7ms	remaining: 1m 16s
100:	learn: 0.7722053	test: 0.6754997	best: 0.6797190 (95)	total: 5.89s	remaining: 52.4s
200:	learn: 0.8650622	test: 0.6922212	best: 0.6922212 (200)	total: 8.89s	remaining: 35.3s
300:	learn: 0.9300335	test: 0.6990932	best: 0.7028816 (268)	total: 11.8s	remaining: 27.4s
400:	learn: 0.9682407	test: 0.7029946	best: 0.7096300 (347)	total: 18.2s	remaining: 27.2s
500:	learn: 0.9883557	test: 0.7120191	best: 0.7120191 (499)	total: 23.3s	remaining: 23.2s
600:	learn: 0.9979225	test: 0.7010554	best: 0.7120191 (499)	total: 26.2s	remaining: 17.4s
700:	learn: 0.9997034	test: 0.7028363	best: 0.7120191 (499)	total: 29.2s	remaining: 12.4s
800:	learn: 1.0000000	test: 0.6992374	best: 0.7120191 (499)	total: 32.4s	remaining: 8.05s
900:	learn: 1.0000000	test: 0.7022733	best: 0.7120191 (499)	total: 37.9s	remaining: 4.16s
999:	learn: 1.0000000	test: 0.6983021	best: 0.7120191 (499)	total: 40.8s	

In [41]:
from sklearn.metrics import classification_report

y_val_pred = clf.predict(X_val)
print(classification_report(y_val, y_val_pred))

                         precision    recall  f1-score   support

              Demografi       0.00      0.00      0.00        10
                Ekonomi       0.80      0.58      0.67        55
               Geografi       0.00      0.00      0.00         1
               Ideologi       0.82      0.18      0.30        49
Pertahanan dan Keamanan       0.78      0.37      0.50        49
                Politik       0.77      0.95      0.85       580
          Sosial Budaya       0.49      0.24      0.32        76
       Sumber Daya Alam       0.41      0.29      0.34        24

               accuracy                           0.75       844
              macro avg       0.51      0.33      0.37       844
           weighted avg       0.73      0.75      0.71       844



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [42]:
from sklearn.metrics import balanced_accuracy_score

balanced_accuracy_score(y_val, y_val_pred)

0.32620290299752963

In [43]:
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold


model_1 = CatBoostClassifier(task_type="GPU", verbose=100, random_state=42)

cv_method = RepeatedStratifiedKFold(n_splits=5, n_repeats=1, random_state=42)
cv_results_model_1 = cross_val_score(estimator=clf, X=train_df_vectors, y=train_df['label'], cv=cv_method,
                                     scoring='balanced_accuracy')

print("Model 1 - Balanced Accuracy scores: ")
print(cv_results_model_1)
print("Mean: ")
print(cv_results_model_1.mean().round(3))

Learning rate set to 0.083181
0:	learn: 0.6247920	total: 73.6ms	remaining: 1m 13s
100:	learn: 0.7428014	total: 5.01s	remaining: 44.6s
200:	learn: 0.8185451	total: 8.84s	remaining: 35.1s
300:	learn: 0.8809350	total: 11.7s	remaining: 27.2s
400:	learn: 0.9226334	total: 14.6s	remaining: 21.9s
500:	learn: 0.9560857	total: 18.7s	remaining: 18.7s
600:	learn: 0.9756733	total: 23.2s	remaining: 15.4s
700:	learn: 0.9865639	total: 26s	remaining: 11.1s
800:	learn: 0.9949475	total: 28.9s	remaining: 7.19s
900:	learn: 0.9985160	total: 32s	remaining: 3.51s
999:	learn: 0.9997033	total: 37.5s	remaining: 0us
Learning rate set to 0.083181
0:	learn: 0.6223115	total: 35.2ms	remaining: 35.2s
100:	learn: 0.7456890	total: 3.12s	remaining: 27.8s
200:	learn: 0.8191112	total: 6.06s	remaining: 24.1s
300:	learn: 0.8754905	total: 9.78s	remaining: 22.7s
400:	learn: 0.9183367	total: 14.6s	remaining: 21.9s
500:	learn: 0.9484441	total: 17.5s	remaining: 17.4s
600:	learn: 0.9765871	total: 20.4s	remaining: 13.5s
700:	learn:

In [ ]:
# Similarity search
sims = []
for n, i in enumerate(tweet_corpus):
  _sim = model.similarity(tweet, i)
  sims.append([n, _sim])
print(f'anchor tweet: {tweet}\n')
for m, (n, s) in enumerate(sorted(sims, key=lambda x: x[1], reverse=True)):
  print(f' - top {m}: {tweet_corpus[n]}\n - similaty: {s}\n')

anchor tweet: I will never understand the decision making of the people of Alabama. Their new Senator is a definite downgrade. You have served with honor.  Well done.

 - top 0: Is this a confirmation from Q that Lin is leaking declassified intelligence to the public? I believe so. If @realDonaldTrump didn’t approve of what @LLinWood is doing he would have let us know a lonnnnnng time ago. I’ve always wondered why Lin’s Twitter handle started with “LLin” https://t.co/0G7zClOmi2
 - similaty: 1.0787510714776494

 - top 1: Tomorrow is my last day as Senator from Alabama.  I believe our opportunities are boundless when we find common ground. As we swear in a new Congress &amp; a new President, demand from them that they do just that &amp; build a stronger, more just society.  It’s been an honor to serve you.The mask cult can’t ever admit masks don’t work because their ideology is based on feeling like a “good person”  Wearing a mask makes them a “good person” &amp; anyone who disagrees w/t

## Use Custom Model
To use an other model from local/huggingface modelhub, one can simply provide the model path/alias to the `load` function.

`tweetnlp.load('task', model='model-path/alias')`

Or any classification model can be used without specifying the task.


In [ ]:
# other task eg) NER
model = tweetnlp.load_model('ner', model_name='tner/twitter-roberta-base-2019-90m-tweetner7-continuous')
model.ner("Jacob Collier is a Grammy-awarded English artist from London.")

Downloading:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/385 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/798k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/456k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/239 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/496M [00:00<?, ?B/s]

[{'type': 'person', 'entity': 'Jacob Collier'},
 {'type': 'location', 'entity': ' London'}]

## Fine-tuning Language Model with TweetNLP
TweetNLP provides an easy interface to fine-tune language models on the datasets supported by HuggingFace for model hosting/fine-tuning with [RAY TUNE](https://docs.ray.io/en/latest/tune/index.html) for parameter search.
- Supported Tasks: `sentiment`, `offensive`, `irony`, `hate`, `emotion`, `topic_classification`



In [54]:
# load dataset
dataset, label_to_id = tweetnlp.load_dataset("topic_classification")

  0%|          | 0/13 [00:00<?, ?it/s]

In [50]:
train_df.label.unique().tolist()

['Sumber Daya Alam',
 'Politik',
 'Demografi',
 'Pertahanan dan Keamanan',
 'Ideologi',
 'Ekonomi',
 'Sosial Budaya',
 'Geografi']

In [66]:
label_to_id = {v: k for k, v in enumerate(train_df.label.unique().tolist())}
label_to_id

{'Sumber Daya Alam': 0,
 'Politik': 1,
 'Demografi': 2,
 'Pertahanan dan Keamanan': 3,
 'Ideologi': 4,
 'Ekonomi': 5,
 'Sosial Budaya': 6,
 'Geografi': 7}

In [68]:
from datasets import Dataset

train_df_finetune = train_df.copy()
train_df_finetune["label_name"] = train_df_finetune["label"]
train_df_finetune['label'] = train_df_finetune['label_name'].apply(lambda x: label_to_id[x])
dataset = Dataset.from_pandas(train_df_finetune)
dataset

Dataset({
    features: ['text', 'label', 'label_name'],
    num_rows: 4218
})

In [63]:
dataset["train_coling2022"].to_pandas()

                                                   text        date  label  \
0     The {@Clinton LumberKings@} beat the {@Cedar R...  2019-09-08      4   
1     Major sidenote of the MSU game: How about the ...  2019-09-08      4   
2     I would rather hear Eli Gold announce this Aub...  2019-09-08      4   
3     Someone take my phone away, I’m trying to not ...  2019-09-08      4   
4     A year ago, Louisville struggled to beat an FC...  2019-09-08      4   
...                                                 ...         ...    ...   
3593  #NowPlaying Leaving in the Morning - Chicken D...  2020-08-30      2   
3594  A little bit of humour for #politas this morni...  2020-08-30      3   
3595  TheRealDimmak is live on YouTube (Call of Duty...  2020-08-30      4   
3596  . {{USERNAME}} :  RPT #Sinopec posts first hal...  2020-08-30      1   
3597  Facilitating 1st grade over here this morning....  2020-08-31      3   

                    label_name                   id  
0        

In [47]:
dataset

DatasetDict({
    test_2020: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 376
    })
    test_2021: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 1693
    })
    train_2020: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 2858
    })
    train_2021: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 1516
    })
    train_all: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 4374
    })
    validation_2020: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 352
    })
    validation_2021: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 189
    })
    train_random: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 2830
    })
    validati

In [69]:
import logging
import tweetnlp
from pprint import pprint

logging.basicConfig(format='%(asctime)s %(levelname)-8s %(message)s', level=logging.INFO, datefmt='%Y-%m-%d %H:%M:%S')

# # an examples for model prediction
# sample = [
#     "How many more days until opening day? 😩"
#     "All two of them taste like ass.",
#     "If you wanna look like a badass, have drama on social media",
#     "Whoever just unfollowed me you a bitch",
#     "I love swimming for the same reason I love meditating...the feeling of weightlessness.",
#     "Beautiful sunset last night from the pontoon @ Tupper Lake, New York",
#     'Jacob Collier is a Grammy-awarded English artist from London.'
# ]

# set language model and task
language_model = 'cardiffnlp/tweet-topic-latest-multi'
task = "topic_classification"

# # load dataset
# dataset, label_to_id = tweetnlp.load_dataset(task)

# load trainer
trainer_class = tweetnlp.load_trainer(task)

# define trainer
trainer = trainer_class(
    language_model=language_model,
    dataset=dataset,
    label_to_id=label_to_id,
    max_length=128,
    split_train='train',
    split_test='test',
    output_dir=f'model_ckpt/test'
)

RuntimeError: Error(s) in loading state_dict for RobertaForSequenceClassification:
	size mismatch for classifier.out_proj.weight: copying a param with shape torch.Size([19, 768]) from checkpoint, the shape in current model is torch.Size([8, 768]).
	size mismatch for classifier.out_proj.bias: copying a param with shape torch.Size([19]) from checkpoint, the shape in current model is torch.Size([8]).
	You may consider adding `ignore_mismatched_sizes=True` in the model `from_pretrained` method.

In [ ]:
# train
trainer.train(down_sample_size_train=1000, ray_result_dir="ray_results/test")

# save model checkpoint
trainer.save_model()

In [ ]:
# model evaluation
metrics = trainer.evaluate()
pprint(metrics)

In [ ]:
# sample prediction
output = trainer.predict(sample)
pprint(f"Sample Prediction: {language_model} ({task})")
for s, p in zip(sample, output):
    pprint(s)
    pprint(p)